# 01. EEG Emotion Recognition

Train a classical machine-learning model on the EEG emotion dataset and save the scaler, label encoder, and classifier artifacts used by the backend.


In [1]:
from pathlib import Path
import sys

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "NeuroSense" / "webdev" / "backend").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root for this notebook.")

PROJECT_ROOT = _find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "NeuroSense" / "notebooks"
BACKEND_DIR = PROJECT_ROOT / "NeuroSense" / "webdev" / "backend"

for path in (NOTEBOOKS_DIR, BACKEND_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from notebook_support import bootstrap_notebook

ctx = bootstrap_notebook(PROJECT_ROOT)
DATASETS_DIR = ctx["datasets_dir"]
ARTIFACTS_DIR = ctx["artifacts_dir"]
CACHE_DIR = ctx["cache_dir"]
RANDOM_STATE = ctx["random_state"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Datasets directory: {DATASETS_DIR}")
print(f"Artifacts directory: {ARTIFACTS_DIR}")


Project root: /Users/devashishsingh/Desktop/human emotion recognition system
Datasets directory: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/datasets
Artifacts directory: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/artifacts


In [2]:
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.svm import SVC

eeg_path = DATASETS_DIR / "eeg" / "eeg" / "emotions.csv"
if not eeg_path.exists():
    raise FileNotFoundError(f"Missing EEG dataset: {eeg_path}")

df = pd.read_csv(eeg_path)
if "label" not in df.columns:
    raise ValueError("Expected a 'label' column in the EEG dataset.")

feature_columns = [column for column in df.columns if column != "label"]
X = df[feature_columns].select_dtypes(include=[np.number]).fillna(0.0)
y = df["label"].astype(str)

print("Dataset shape:", df.shape)
print("Label counts:", y.value_counts().to_dict())


Dataset shape: (2132, 2549)
Label counts: {'NEUTRAL': 716, 'NEGATIVE': 708, 'POSITIVE': 708}


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x=y, ax=axes[0], order=sorted(y.unique()), palette="viridis")
axes[0].set_title("EEG label distribution")
axes[0].tick_params(axis="x", rotation=20)

axes[1].hist(X.iloc[:, 0], bins=20, color="#2d8fdd")
axes[1].set_title(f"Distribution of {feature_columns[0]}")
plt.tight_layout()
plt.show()

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X.values,
    y_enc,
    test_size=0.2,
    stratify=y_enc,
    random_state=RANDOM_STATE,
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

candidate_models = {
    "svm_rbf": SVC(kernel="rbf", C=10.0, probability=True, random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(n_estimators=250, random_state=RANDOM_STATE, n_jobs=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
model_rows = []
fitted_models = {}

for name, estimator in candidate_models.items():
    cv_scores = cross_val_score(
        make_pipeline(StandardScaler(), estimator),
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
    )
    fitted_estimator = estimator.fit(X_train_sc, y_train)
    fitted_models[name] = fitted_estimator
    test_accuracy = accuracy_score(y_test, fitted_estimator.predict(X_test_sc))
    model_rows.append(
        {
            "model": name,
            "cv_mean": round(float(cv_scores.mean()), 4),
            "cv_std": round(float(cv_scores.std()), 4),
            "test_accuracy": round(float(test_accuracy), 4),
        }
    )

results_df = pd.DataFrame(model_rows).sort_values(["test_accuracy", "cv_mean"], ascending=False)
best_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_name]
results_df


/var/folders/y1/kwl357md29bd8ggvss23h_yc0000gn/T/ipykernel_59660/1754292123.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(x=y, ax=axes[0], order=sorted(y.unique()), palette="viridis")
/var/folders/y1/kwl357md29bd8ggvss23h_yc0000gn/T/ipykernel_59660/1754292123.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,model,cv_mean,cv_std,test_accuracy
1,random_forest,0.9830,0.0088,0.9859
0,svm_rbf,0.9683,0.0057,0.9766


In [4]:
y_pred = best_model.predict(X_test_sc)
print(f"Selected model: {best_name}")
print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("EEG confusion matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

y_test_bin = label_binarize(y_test, classes=list(range(len(le.classes_))))
y_score = best_model.predict_proba(X_test_sc)

plt.figure(figsize=(7, 5))
for index, class_name in enumerate(le.classes_):
    fpr, tpr, _ = roc_curve(y_test_bin[:, index], y_score[:, index])
    plt.plot(fpr, tpr, label=f"{class_name} (AUC={auc(fpr, tpr):.2f})")
plt.plot([0, 1], [0, 1], "k--")
plt.title("EEG one-vs-rest ROC curves")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.legend()
plt.tight_layout()
plt.show()


Selected model: random_forest
              precision    recall  f1-score   support

    NEGATIVE       0.98      0.99      0.98       142
     NEUTRAL       0.99      1.00      1.00       143
    POSITIVE       0.99      0.97      0.98       142

    accuracy                           0.99       427
   macro avg       0.99      0.99      0.99       427
weighted avg       0.99      0.99      0.99       427



/var/folders/y1/kwl357md29bd8ggvss23h_yc0000gn/T/ipykernel_59660/4265924410.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/y1/kwl357md29bd8ggvss23h_yc0000gn/T/ipykernel_59660/4265924410.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
artifact_dir = ARTIFACTS_DIR / "eeg"
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, artifact_dir / "eeg_model.pkl")
joblib.dump(scaler, artifact_dir / "eeg_scaler.pkl")
joblib.dump(le, artifact_dir / "eeg_label_encoder.pkl")

sample_probs = best_model.predict_proba(X_test_sc[:3])
print("Saved EEG artifacts to:", artifact_dir)
print("Sample probabilities:")
print(pd.DataFrame(sample_probs, columns=le.classes_).round(4))


Saved EEG artifacts to: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/artifacts/eeg
Sample probabilities:
   NEGATIVE  NEUTRAL  POSITIVE
0     0.012      0.5     0.488
1     0.940      0.0     0.060
2     0.992      0.0     0.008
